In [5]:
%cd "E:/src code 2/python 2/KG"
import json
import pandas as pd
import os

from src.index.entity_extractor import EntityExtractor
from src.utils.config_loader import ConfigLoader
from src.llm.gemini import Gemini_LLM
from src.utils.utils import read_file
import os
from src.db.neo4j import GraphManager, Node, Edge
from src.utils.type import TYPE_OF_ENTITY_IN_KG, TYPE_OF_JOB, TYPE_OF_JOB_ENTITY, TYPE_OF_CV, TYPE_OF_EDGE
config = ConfigLoader().get_config_from_file(r"E:\src code 2\python 2\Legal_RAG\config\config.yaml")
llm = Gemini_LLM(config=config)

gm = GraphManager("neo4j://localhost:7687", "neo4j", "123123aA@")
entities = ["Programming Language", "Library", "Software", "Technology", "Task", "Country", "City", "District"]


E:\src code 2\python 2\KG


In [ ]:
folder = "E:/data/job/"
entity_extractor = EntityExtractor(entities=", ".join(entities), llm=llm, cache_folder=folder)
for file in os.listdir(folder):
    if not file.endswith("_v3.txt"): continue
    entities_list, _ = entity_extractor.parse_entities(read_file(folder + file))
    job_name = file[:-4]
    job_node = Node(TYPE_OF_JOB, {"name":job_name})
    gm.add_edge(job_node)
    for e in entities_list:
        e = e.node()
        e.label = TYPE_OF_JOB_ENTITY
        gm.add_node(e)
        gm.add_edge(Edge(job_node, e, TYPE_OF_EDGE))